# E5 · Curvas de contraste (anillos)

**Spec:** [`docs/spec_E5_codex_contrast_curves.md`](../docs/spec_E5_codex_contrast_curves.md)  |  **Bloque:** E · Resultado  |  **Run por defecto:** `ROXs12b_realigned`

Contraste mínimo detectable vs separación por método (sgf/lpm), con inyecciones en grilla de anillos validadas sobre el mapa E1b (Julo et al. 2025 Fig. 10).

| | |
|---|---|
| **Entrada** | Cubos residuales C5/C6 + cubo madre + PSF C1 |
| **Salida (QC/productos)** | `tables/contrast_curve_by_method.csv`, `tables/contrast_injections.csv`, `stages/stage_h05_qc.json` |
| **Consume aguas abajo** | E6 (ROC); notebook 10 de límites de masa (consumidor futuro) |


## Qué hace E5

Inyecta líneas falsas (gaussiana FWHM=LSF × PSF C1) en anillos concéntricos (separaciones × 8 ángulos × contrastes 1e-5→1e-2 relativos al flujo estelar en la banda de línea) y valida cada una con la MISMA cadena de detección de E1b (matched filter + μ̂/σ̂ robustos por anillo, umbral 5σ). Por la linealidad de la sustracción con ŝ fija, el residual base y sus anillos se calculan UNA vez y cada inyección solo aporta su delta — la grilla completa cuesta segundos y la decisión usa la realización de ruido REAL de cada posición (como el paper §3.3.2).

La curva reportada es el contraste con ≥50% de detección sobre los ángulos, con bandas al 25/75%.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs12b_realigned   # o ROXs12b_B_adp para comparar
cd MUSE-accretion-pipeline                    # raíz del repo
python -m musepipe.stages.stage_h05_contrast --run-id $RUN
```

Ligero-moderado (~min: camino delta lineal, grilla congelada en spec).

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage_h05_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'python -m musepipe.stages.stage_h05_contrast --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage_h05_qc.json', RUN_ID)
nb.show(qc, keys=['params.threshold_sigma', 'params.f_star_line', 'checks.v1_curves_written', 'checks.v2_monotonic_trend'], title='E5')


## Evidencia: curva por método


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('E5', 'stages/stage_h05_qc.json'):
        q = nb.load_qc('stages/stage_h05_qc.json', RUN_ID)
        print('F_star(banda línea) =', q['params']['f_star_line'])
        for m, mq in q['methods'].items():
            print(f"  {m}: {mq['n_injections']} inyecciones")
            for e in mq['curve'][:6]:
                print(f"    r={e['separation_px']:5.1f}px  c50={e['contrast_50']}  c25={e['contrast_25']}  c75={e['contrast_75']}")
        print('checks:', q['checks'])


## Plot — curva de contraste 5σ (Fig. 10 del paper)

Contraste al 50% con banda 25/75%, eje x en arcsec (25 mas/px).


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    q = nb.load_qc('stages/stage_h05_qc.json', RUN_ID)
    fig, ax = plt.subplots(figsize=(8.5, 4.4))
    for m, mq in q['methods'].items():
        r = [e['separation_px']*0.025 for e in mq['curve'] if e['contrast_50']]
        c50 = [e['contrast_50'] for e in mq['curve'] if e['contrast_50']]
        lo = [e['contrast_25'] or np.nan for e in mq['curve'] if e['contrast_50']]
        hi = [e['contrast_75'] or np.nan for e in mq['curve'] if e['contrast_50']]
        ax.plot(r, c50, marker='o', ms=3, label=m)
        ax.fill_between(r, lo, hi, alpha=0.2)
    ax.set_yscale('log'); ax.set_xlabel('separación [arcsec]')
    ax.set_ylabel('contraste de línea 5σ'); ax.legend()
    ax.set_title('E5 · curvas de contraste'); fig.tight_layout(); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- Detección con la cadena E1b real (no un detector ad-hoc); grillas congeladas en la spec. · [`docs/spec_E5_codex_contrast_curves.md`](../docs/spec_E5_codex_contrast_curves.md)
- Camino delta lineal exacto (verificado contra fuerza bruta en tests); psfsub opcional con re-sustracción completa. · [`docs/plan_integracion_halosub_julo2025.md`](../docs/plan_integracion_halosub_julo2025.md)


## Checks


In [ ]:
try:
    q = nb.load_qc('stages/stage_h05_qc.json', RUN_ID)
    for k, v in q['checks'].items():
        print(f'  {k}: {v}')
except FileNotFoundError as e:
    print('QC aún no existe para este run:', e)


## Estado

**Pendiente de primera ejecución sobre datos reales** (requiere C5/C6). Núcleo y contrato verificados con tests sintéticos (2026-07-14).
